# EU AI Act -  End-to-End Loan Application Example


This notebook explains the **EU AI Act** using a synthetic AI Loan Recommendation Assistant.

We use only one input file:

`loan_applications.csv`

The notebook keeps the legal/governance flow simple:

```text
DEFINE THE AI SYSTEM
        ↓
IDENTIFY OUR ROLE
        ↓
CHECK PROHIBITED USES
        ↓
CLASSIFY THE RISK
        ↓
MAP THE MAIN OBLIGATIONS
        ↓
OPERATE THE AI WITH CONTROLS
        ↓
TEST FAIRNESS / TRANSPARENCY / RELIABILITY
        ↓
HUMAN OVERSIGHT
        ↓
DOCUMENT EVIDENCE
        ↓
IDENTIFY GAPS
```

> This is a learning example, not legal advice and not a production lending system.

## Why this loan use case is useful

The EU AI Act uses a **risk-based approach**.

A system used to evaluate the **creditworthiness of natural persons** is a sensitive use case and can fall within the high-risk category.

That makes a loan recommendation system useful for learning:

- risk classification
- provider and deployer roles
- risk management
- data governance
- transparency
- human oversight
- technical documentation
- logging and monitoring
- evidence and gap analysis

For this notebook we treat the use case as:

```text
HIGH_RISK_REVIEW
```

A real system would require qualified legal/compliance review before final classification.

##  EU AI Act Mapping

| Area | What we do in this notebook |
|---|---|
| Scope | Define the AI system and intended purpose |
| Role | Identify provider / deployer responsibilities |
| Prohibited Practices | Perform a simple screening |
| Risk Classification | Treat the lending use case as high-risk review |
| Risk Management | Identify main risks |
| Data Governance | Control which data is sent to the LLM |
| Transparency | Check explanations |
| Human Oversight | Keep a human as final decision-maker |
| Reliability | Measure recommendation quality |
| Logging | Save important evidence |
| Gap Analysis | Identify what is still missing |

This is the complete flow.

## Installation

```bash
pip install pandas langchain-openai openai python-dotenv
```

Create a `.env` file:

```text
OPENAI_API_KEY=your_openai_api_key
```

# Step 1 - Load and understand the loan dataset

The dataset contains:

- customer ID
- age
- annual income
- credit score
- gender
- region
- existing debt
- historical approval

The `approved` column is used only as reference evidence.

In [ ]:
import pandas as pd
df = pd.read_csv("loan_applications.csv")
print("Rows and columns:",df.shape)
print(df.head())
print("\nMissing values:")
print(df.isnull().sum())


# Step 2 - Define the AI system and intended purpose

EU AI Act classification starts with the **intended purpose**.

For this example, the AI system:

- supports internal loan reviewers
- produces APPROVE / REJECT recommendations
- does not make the final decision
- affects individual loan applicants

This definition is important because legal obligations depend on how the AI is actually intended to be used.

In [ ]:
ai_system = {"name":"AI Loan Recommendation Assistant","purpose":"Provide a loan recommendation to internal lending staff","users":"Loan reviewers","affected_people":"Loan applicants","final_decision":"Human loan reviewer","model":"OpenAI chat model through LangChain"}
print(ai_system)


# Step 3 - Identify our role

The EU AI Act assigns different responsibilities depending on the organization's role.

Common roles include:

- **Provider** - develops or offers an AI system under its own name
- **Deployer** - uses an AI system in its organization
- **Importer / Distributor** - brings or distributes AI systems in the EU
- **GPAI Provider** - provides a general-purpose AI model

For this simple example, we assume the lending organization is both:

```text
Provider of the loan application
+
Deployer of the application
```

The underlying LLM is supplied by a third-party model provider.

In [ ]:
roles = {"application_provider":"Example Lending Organization","application_deployer":"Example Lending Organization","underlying_model_provider":"Third-party LLM provider","final_authority":"Human loan reviewer"}
print(roles)


# Step 4 - Screen for prohibited practices

Before checking high-risk obligations, we first ask:

> Is the intended use obviously prohibited?

For this loan use case we check for examples such as:

- social scoring
- manipulative practices
- exploitation of vulnerable people
- prohibited biometric uses

This notebook records only a preliminary screening result.

In [ ]:
prohibited_checks = {"social_scoring":False,"manipulative_practice":False,"vulnerability_exploitation":False,"prohibited_biometric_use":False}
prohibited_status = "REVIEW_REQUIRED" if any(prohibited_checks.values()) else "NO_OBVIOUS_PROHIBITED_USE"
print("Prohibited-practice screen:",prohibited_status)


# Step 5 - Classify the use case

Because the system supports evaluation of an individual's creditworthiness, we treat it as a **high-risk review** use case for this demonstration.

The purpose of classification is to determine how much governance and evidence is required.

In [ ]:
risk_classification = {"use_case":"Loan / credit recommendation for an individual","classification":"HIGH_RISK_REVIEW","reason":"The system supports an important financial decision affecting a natural person"}
print(risk_classification)


# Step 6 - Identify the main obligations

For a high-risk AI system, the EU AI Act expects lifecycle controls.

We keep the obligation list simple:

- risk management
- data governance
- technical documentation
- logging
- transparency
- human oversight
- reliability / robustness
- cybersecurity
- post-deployment monitoring

The notebook demonstrates some of these directly and clearly marks the remaining gaps.

In [ ]:
obligations = ["Risk management","Data governance","Technical documentation","Logging and traceability","Transparency","Human oversight","Accuracy and reliability","Cybersecurity","Post-deployment monitoring"]
for item in obligations:
    print("-",item)


# Step 7 - Identify the main AI risks

For the lending assistant, we identify a small number of realistic risks:

- unfair recommendations
- incorrect recommendations
- hallucinated lending rules
- sensitive-data exposure
- excessive automation
- prompt injection / misuse

This becomes the basis for our controls and evaluations.

In [ ]:
risks = ["Unfair recommendations","Incorrect recommendations","Hallucinated lending rules","Sensitive data exposure","Too much reliance on AI","Prompt injection or misuse"]
for item in risks:
    print("-",item)


# Step 8 - Apply data-governance controls

The dataset contains fields such as gender, age, and region.

For the actual LLM recommendation, we use only:

- annual income
- credit score
- existing debt

We do **not** send gender, age, or region to the LLM.

Gender remains in the dataset only for fairness auditing after the AI recommendation is produced.

In [ ]:
decision_features = ["annual_income","credit_score","existing_debt"]
excluded_from_llm = ["age","gender","region"]
print("Decision features:",decision_features)
print("Excluded from LLM:",excluded_from_llm)


# Step 9 - Generate loan recommendations with LangChain OpenAI

The LLM returns only:

```text
APPROVE
or
REJECT
```

Only 20 rows are used to keep the demonstration simple.

The LLM remains a recommendation tool. It does not make the final lending decision.

In [ ]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
load_dotenv()
llm = ChatOpenAI(model="gpt-4o-mini",temperature=0)
sample = df.head(20).copy()
def recommend(row):
    prompt = f'''This is a synthetic lending exercise.
Use only:
Annual Income: {row["annual_income"]}
Credit Score: {row["credit_score"]}
Existing Debt: {row["existing_debt"]}
Do not use age, gender, region, or any protected attribute.
Return only APPROVE or REJECT.'''
    return llm.invoke(prompt).content.strip().upper()
sample["llm_decision"] = sample.apply(recommend,axis=1)
sample["llm_prediction"] = sample["llm_decision"].map({"APPROVE":1,"REJECT":0})
print(sample[["customer_id","gender","approved","llm_decision"]])


# Step 10 - Measure recommendation quality

One high-risk obligation is to evaluate whether the system performs reliably.

We compare the LLM recommendation with the historical `approved` column.

This is only a reference metric.

It does not mean the historical decision is automatically correct.

In [ ]:
valid = sample.dropna(subset=["llm_prediction"])
agreement = round((valid["llm_prediction"]==valid["approved"]).mean(),3)
print("Agreement with historical approvals:",agreement)


# Step 11 - Measure fairness

Gender was not used to make the recommendation.

After the recommendation is generated, we use gender only to audit whether positive recommendation rates differ between groups.

For this simple learning example:

- ratio >= 0.80 → PASS
- ratio < 0.80 → REVIEW

In [ ]:
group_rates = sample.groupby("gender")["llm_prediction"].mean().round(3)
valid_rates = group_rates.dropna()
fairness_ratio = round(min(valid_rates)/max(valid_rates),3) if len(valid_rates)>=2 and max(valid_rates)>0 else 0
fairness_status = "PASS" if fairness_ratio>=0.80 else "REVIEW"
print(group_rates)
print("Fairness ratio:",fairness_ratio)
print("Fairness status:",fairness_status)


# Step 12 - Check transparency

A human reviewer should be able to understand the recommendation.

For one applicant, we ask the model for a short explanation based only on the supplied financial facts.

This demonstrates the idea of transparency without making the notebook complicated.

In [ ]:
row = sample.iloc[0]
explanation_prompt = f'''This is a synthetic lending exercise.
Applicant facts:
Annual Income: {row["annual_income"]}
Credit Score: {row["credit_score"]}
Existing Debt: {row["existing_debt"]}
Explain the recommendation in 2 short sentences.
Use only these facts.
Do not use protected attributes or invent policy.'''
explanation = llm.invoke(explanation_prompt).content
print("Applicant:",row["customer_id"])
print("Recommendation:",row["llm_decision"])
print("Explanation:",explanation)


# Step 13 - Apply human oversight

For this high-impact use case:

- AI produces only a recommendation
- a human reviewer makes the final decision
- the human can override the AI
- fairness issues trigger additional review

This is one of the most important controls in the notebook.

In [ ]:
human_oversight = {"final_authority":"Human loan reviewer","ai_can_make_final_decision":False,"human_can_override":True,"additional_review":fairness_status=="REVIEW"}
print(human_oversight)


# Step 14 - Create simple technical documentation and logging

High-risk governance requires evidence.

We record:

- system name
- intended purpose
- role
- risk classification
- model
- input factors
- sample size
- reliability metric
- fairness metric
- human oversight

We also create a simple traceability log of the AI recommendations.

In [ ]:
technical_documentation = {"system":ai_system["name"],"purpose":ai_system["purpose"],"role":"Provider + Deployer","classification":risk_classification["classification"],"model":ai_system["model"],"decision_features":", ".join(decision_features),"sample_size":len(sample),"agreement":agreement,"fairness_ratio":fairness_ratio,"fairness_status":fairness_status,"final_authority":human_oversight["final_authority"]}
log = sample[["customer_id","approved","llm_decision"]].copy()
log["system_version"] = "demo-v1"
log["timestamp_utc"] = pd.Timestamp.utcnow().isoformat()
print(technical_documentation)
print(log.head())


# Step 15 - Identify gaps and make the final governance decision

A useful compliance notebook should not pretend everything is complete.

For this simple demonstration we mark:

```text
Cybersecurity testing → GAP
Post-deployment monitoring → GAP
Formal legal/compliance review → GAP
```

That means the system is **not ready for production**.

This is an important EU AI Act lesson:

> Passing a model test is not the same as completing high-risk compliance.

In [ ]:
gaps = ["Cybersecurity testing not completed","Post-deployment monitoring not implemented","Formal legal/compliance review not completed"]
final_status = "NOT_READY_FOR_PRODUCTION" if len(gaps)>0 else "READY_FOR_FINAL_COMPLIANCE_REVIEW"
print("Open gaps:")
for item in gaps:
    print("-",item)
print("\nFinal status:",final_status)
sample.to_csv("eu_ai_act_loan_results.csv",index=False)
log.to_csv("eu_ai_act_traceability_log.csv",index=False)
pd.DataFrame([technical_documentation]).to_csv("eu_ai_act_technical_documentation.csv",index=False)
pd.DataFrame({"gap":gaps}).to_csv("eu_ai_act_open_gaps.csv",index=False)
print("\nEvidence files saved.")


# Final EU AI Act Checklist

## DEFINE THE SYSTEM
- [ ] Intended purpose documented
- [ ] Users and affected people identified
- [ ] Final human authority defined

## IDENTIFY THE ROLE
- [ ] Provider role reviewed
- [ ] Deployer role reviewed
- [ ] Third-party model provider identified

## CLASSIFY THE RISK
- [ ] Prohibited-practice screen completed
- [ ] High-risk classification reviewed
- [ ] Legal/compliance review planned

## RISK MANAGEMENT
- [ ] Main AI risks identified
- [ ] Risk treatments defined
- [ ] Human oversight included

## DATA GOVERNANCE
- [ ] Decision features documented
- [ ] Protected attributes controlled
- [ ] Data-quality issues checked

## TRANSPARENCY
- [ ] Human reviewer sees the AI recommendation
- [ ] Explanation available
- [ ] Limitations documented

## RELIABILITY
- [ ] Recommendation quality measured
- [ ] Fairness measured
- [ ] Important failure modes reviewed

## HUMAN OVERSIGHT
- [ ] AI is not the final decision-maker
- [ ] Human can override the AI
- [ ] Review triggers defined

## DOCUMENTATION & LOGGING
- [ ] Technical documentation created
- [ ] AI recommendations logged
- [ ] System version recorded
- [ ] Governance evidence retained

## GAPS BEFORE PRODUCTION
- [ ] Cybersecurity testing completed
- [ ] Post-deployment monitoring implemented
- [ ] Formal legal/compliance review completed

## The easiest way to remember the EU AI Act flow

```text
1. What does the AI do?
        ↓
2. What role do we play?
        ↓
3. Is the use prohibited?
        ↓
4. Is it high-risk?
        ↓
5. What obligations apply?
        ↓
6. Implement controls
        ↓
7. Test and document evidence
        ↓
8. Keep a human in control
        ↓
9. Close compliance gaps
        ↓
10. Deploy and monitor
```

That is the core risk-based governance idea demonstrated by this notebook.